# M03B · The Four Pillars of OOP & Access Modifiers
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2024-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M03_OOP/solutions/M03B_Four_Pillars_and_Access_Modifiers_Solutions.ipynb)

⏱ ~120 min · Difficulty: ★★★ · Prerequisites: M03A (Classes, Methods, Dunders)

---


## 1. Learning Outcomes

By the end of this notebook, you will be able to:
1. **Differentiate** between public, protected, and private access modifiers in Python and demonstrate name mangling mechanics under the hood.
2. **Implement encapsulation** using `@property` getters, setters, and deleters to enforce biological data invariants.
3. **Construct** class hierarchies using single and multiple inheritance, `super()`, and Method Resolution Order (MRO).
4. **Implement polymorphism and duck typing** to process heterogeneous biological sequences and assay models uniformly.
5. **Design** robust abstract base classes using `abc.ABC` and `@abstractmethod` to define strict interface contracts.
6. **Analyze** core conceptual OOP design principles in bioinformatics software.

---


## 2. Prerequisites

Before starting this notebook, ensure you understand:
- Defining classes, instance methods, and `__init__` constructors (M03A).
- Python dunder methods (`__len__`, `__str__`, `__eq__`).
- Biological sequence representations: nucleotides, amino acids, codons, and genomic features.

---


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path
from abc import ABC, abstractmethod

# Detect whether we are running in Google Colab or locally
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root
    ROOT = Path.cwd().resolve()
    while not (ROOT / "data").exists() and ROOT.parent != ROOT:
        ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M03B"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python interpreter : {sys.version.split()[0]}")
print(f"Repository ROOT    : {ROOT.resolve()}")
print(f"DATA_DIR exists    : {DATA_DIR.exists()} ({DATA_DIR})")
print(f"OUT_DIR ready      : {OUT_DIR.exists()} ({OUT_DIR})")


## 4. Concept Sections

### 4.1 Concept 1: Encapsulation & Data Invariants in Biology

**Encapsulation** is the bundling of data (attributes) and the methods that operate on that data into a single unit, while restricting direct access to the internal representation of the object.

#### Why Encapsulation Matters in Bioinformatics
In biological software, corrupted state leads to disastrous analytical errors:
- If external code can set `protein.sequence = "ATGC"`, non-amino acid characters will corrupt structural modeling and mass spectrometry calculations.
- If a patient record's viral load can be set to `-500.0`, epidemiological models produce nonsensical negative transmission probabilities.
- Encapsulation ensures that an object maintains complete authority over its own internal invariants.


In [ ]:
# Worked Example 4.1: Encapsulating a Protein sequence
class ProteinRecord:
    AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWY")

    def __init__(self, accession: str, sequence: str):
        self.accession = accession
        self._set_sequence(sequence)

    def _set_sequence(self, sequence: str):
        clean_seq = sequence.strip().upper()
        invalid = set(clean_seq) - self.AMINO_ACIDS
        if invalid:
            raise ValueError(f"Invalid amino acid characters: {invalid}")
        self._sequence = clean_seq

    def get_sequence(self) -> str:
        return self._sequence

    def length(self) -> int:
        return len(self._sequence)

p = ProteinRecord("NP_000198.1", "MALWMRLLPLLALLALWGPDPAAA")
print(f"Protein: {p.accession} | Length: {p.length()} aa")

try:
    ProteinRecord("BAD_001", "MALWMRZ123")  # Contains Z and numbers
except ValueError as e:
    print(f"Encapsulation caught invalid sequence: {e}")


#### ✏️ Try it 4.1: Encapsulation Check
Verify that `p.get_sequence()` returns the validated uppercase sequence.


In [ ]:
# Try it 4.1: Protein sequence verification
assert p.get_sequence() == "MALWMRLLPLLALLALWGPDPAAA"
assert p.length() == 24
print("Try it 4.1 passed ✔")


#### ⚠️ Common Mistakes with Encapsulation
- **Exposing internal mutable collections directly**: Returning `self._mutations_list` allows external code to call `.append()` or `.clear()` directly, bypassing all validation. Return a copy (`list(self._mutations_list)`) or a tuple instead.
- **Relying on security assumptions**: Encapsulation is a software engineering tool to prevent accidental state corruption, not a cybersecurity barrier against malicious actors.


### 4.2 Concept 2: Access Modifiers & Name Mangling

Unlike languages such as C++ or Java, Python does not have keywords like `public`, `private`, or `protected`. Instead, Python uses **naming conventions**:

#### The Access Modifier Comparison Table
| Modifier | Syntax | Accessible from | Enforced? |
|---|---|---|---|
| **Public** | `self.name` | Anywhere | — (Freely accessible) |
| **Protected** | `self._name` | Class + subclasses (by convention) | **No** — convention only |
| **Private** | `self.__name` | Class only (name-mangled to `_ClassName__name`) | **Partially** (via compiler name mangling) |

#### Python Philosophy: "We Are All Consenting Adults Here"
Python's creator, Guido van Rossum, chose not to build rigid access locks. The single underscore (`_protected`) is an explicit message to developers: *"This is internal implementation detail. Access it at your own risk."*

#### Name Mangling Under the Hood
When Python encounters an attribute with **two leading underscores** and at most one trailing underscore (`self.__name`), the compiler rewrites the identifier to:
`_ClassName__name`

This prevents accidental name collisions between parent and child classes during inheritance!


In [ ]:
# Worked Example 4.2: Access modifiers in clinical patient data
class PatientSample:
    def __init__(self, sample_id: str, lab_batch: str, ssn: str):
        self.sample_id = sample_id          # Public
        self._lab_batch = lab_batch         # Protected (by convention)
        self.__ssn = ssn                    # Private (name-mangled)

    def get_masked_ssn(self) -> str:
        """Safe public access to private data."""
        return "***-**-" + self.__ssn[-4:]

patient = PatientSample("SMPL-8821", "BATCH_2024_Q3", "123-45-6789")

# 1. Public attribute: accessible anywhere
print("Public sample_id      :", patient.sample_id)

# 2. Protected attribute: accessible, but signals internal use
print("Protected _lab_batch  :", patient._lab_batch)

# 3. Private attribute: direct access fails!
try:
    print(patient.__ssn)
except AttributeError as e:
    print(f"Direct private access failed: {e}")

# 4. Accessing through public method
print("Masked SSN            :", patient.get_masked_ssn())

# 5. Proof of Name Mangling: inspecting the internal dictionary
print("Mangled name in dict  :", patient._PatientSample__ssn)
print("Keys in patient dict  :", list(patient.__dict__.keys()))


#### ✏️ Try it 4.2: Verify Name Mangling in `__dict__`
Confirm that `"_PatientSample__ssn"` exists in `patient.__dict__` and that `"__ssn"` does not.


In [ ]:
# Try it 4.2: Name mangling check
assert "_PatientSample__ssn" in patient.__dict__
assert "__ssn" not in patient.__dict__
assert patient._PatientSample__ssn == "123-45-6789"
print("Try it 4.2 passed ✔")


#### ⚠️ Common Mistakes with Access Modifiers
- **Thinking `__private` is secure encryption**: Name mangling is NOT cryptography. Anyone can still read `patient._PatientSample__ssn`.
- **Overusing double underscores**: In general Python programming, single underscore `_protected` is preferred 95% of the time. Double underscores should be reserved for preventing namespace collisions in complex inheritance hierarchies.


### 4.3 Concept 3: Pythonic Properties (`@property`)

In languages like Java or C++, developers write explicit boilerplate getters and setters (`get_gc()`, `set_gc()`). In Python, this is considered unpythonic.

Python provides the `@property` decorator, which allows a method to be accessed with **clean attribute syntax** (`obj.attribute`) while retaining the power of a getter, setter, and deleter:

```python
class Seq:
    @property
    def gc(self):
        # Read-only computed property!
        return self._calc_gc()
```

#### Components of `@property`
1. **Getter (`@property`)**: Defines the attribute read behavior.
2. **Setter (`@prop_name.setter`)**: Validates and controls assignments (`obj.prop_name = value`).
3. **Deleter (`@prop_name.deleter`)**: Handles cleanup when `del obj.prop_name` is called.
4. **Read-Only Properties**: By defining `@property` without a corresponding `@setter`, you create a strictly **read-only** computed attribute!


In [ ]:
# Worked Example 4.3: Validating biological sequences with @property
class BioSequence:
    VALID_NUCLEOTIDES = set("ACGTN")

    def __init__(self, seq_id: str, seq: str):
        self.seq_id = seq_id
        # Triggers the property setter below!
        self.seq = seq

    @property
    def seq(self) -> str:
        """Getter: returns the uppercase sequence."""
        return self._seq

    @seq.setter
    def seq(self, new_seq: str):
        """Setter: validates sequence characters before updating."""
        clean = new_seq.strip().upper()
        invalid = set(clean) - self.VALID_NUCLEOTIDES
        if invalid:
            raise ValueError(f"Invalid nucleotide(s): {invalid} in '{clean}'")
        self._seq = clean

    @seq.deleter
    def seq(self):
        """Deleter: resets sequence to empty string."""
        self._seq = ""

    @property
    def gc_content(self) -> float:
        """Read-only computed property: GC percentage."""
        if not self._seq:
            return 0.0
        gc = self._seq.count("G") + self._seq.count("C")
        return round((gc / len(self._seq)) * 100.0, 2)

dna = BioSequence("probe_01", "atgcgt")
print(f"Sequence via getter  : {dna.seq}")
print(f"Read-only gc_content : {dna.gc_content}%")

# Update sequence via setter
dna.seq = "ggggcccc"
print(f"Updated sequence     : {dna.seq} | New GC: {dna.gc_content}%")

# Test invalid setter
try:
    dna.seq = "ATGCZ"
except ValueError as e:
    print(f"Setter caught error  : {e}")


#### ✏️ Try it 4.3: Read-Only Property Test
Attempting to directly assign to `dna.gc_content = 50.0` should raise an `AttributeError`. Verify this behavior.


In [ ]:
# Try it 4.3: Read-only property test
raised_attr_error = False
try:
    dna.gc_content = 50.0
except AttributeError:
    raised_attr_error = True

assert raised_attr_error, "Read-only property without setter must raise AttributeError"
print("Try it 4.3 passed ✔")


#### ⚠️ Common Mistakes with `@property`
- **Infinite Recursion Bug**: Writing `def seq(self): return self.seq` or `self.seq = new_seq` inside the setter. This endlessly calls the getter/setter itself until Python crashes with `RecursionError: maximum recursion depth exceeded`. You must store and access the value in a backing attribute, such as `self._seq`.
- **Forgetting `@property` before `@x.setter`**: The `@property` getter must always be defined first.


### 4.4 Concept 4: Inheritance, `super()`, and Method Resolution Order (MRO)

**Inheritance** allows a child (derived) class to inherit attributes and methods from a parent (base) class, modeling an **"is-a"** relationship:
- A `Bacterium` **is an** `Organism`.
- A `Plasmid` **is a** `DNASequence`.

#### The `super()` Function
`super().__init__(...)` delegates initialization to the parent class, ensuring base class state is properly established without hardcoding the parent class name.

#### Method Resolution Order (MRO)
In Python, multiple inheritance is resolved using the **C3 Linearization Algorithm**. You can inspect any class's resolution chain via:
`ClassName.mro()` or `ClassName.__mro__`
Python searches for methods from left to right, depth-first, ensuring subclasses are always checked before their parent classes.


In [ ]:
# Worked Example 4.4: Biological organism hierarchy with inheritance
class Organism:
    def __init__(self, scientific_name: str, domain: str):
        self.scientific_name = scientific_name
        self.domain = domain

    def description(self) -> str:
        return f"{self.scientific_name} [Domain: {self.domain}]"

class Bacterium(Organism):
    def __init__(self, scientific_name: str, gram_stain: str):
        # Call parent constructor via super()
        super().__init__(scientific_name=scientific_name, domain="Bacteria")
        self.gram_stain = gram_stain  # "Positive" or "Negative"

    # Override parent method
    def description(self) -> str:
        base_desc = super().description()
        return f"{base_desc} | Gram {self.gram_stain}"

class Eukaryote(Organism):
    def __init__(self, scientific_name: str, chromosome_count: int):
        super().__init__(scientific_name=scientific_name, domain="Eukaryota")
        self.chromosome_count = chromosome_count

    def description(self) -> str:
        base_desc = super().description()
        return f"{base_desc} | 2n = {self.chromosome_count} chromosomes"

ecoli = Bacterium("Escherichia coli", "Negative")
human = Eukaryote("Homo sapiens", 46)

print(ecoli.description())
print(human.description())
print("Bacterium MRO:", [c.__name__ for c in Bacterium.mro()])


#### ✏️ Try it 4.4: Inspect Class Inheritance
Verify that `isinstance(ecoli, Organism)` and `issubclass(Bacterium, Organism)` are both `True`.


In [ ]:
# Try it 4.4: Inheritance checks
assert isinstance(ecoli, Organism)
assert isinstance(ecoli, Bacterium)
assert not isinstance(ecoli, Eukaryote)
assert issubclass(Bacterium, Organism)
print("Try it 4.4 passed ✔")


#### ⚠️ Common Mistakes with Inheritance
- **Calling parent by name**: Writing `Organism.__init__(self, ...)` instead of `super().__init__(...)`. This breaks cooperative multiple inheritance and diamond hierarchies.
- **Forgetting `super().__init__()`**: Omitting `super().__init__()` leaves parent attributes uninitialized, causing `AttributeError` when accessing parent variables.


### 4.5 Concept 5: Polymorphism & Duck Typing

**Polymorphism** ("many forms") allows different classes to implement the same method interface, enabling client code to process heterogeneous objects uniformly.

#### Duck Typing in Python
> *"If it walks like a duck and quacks like a duck, it's a duck."*

Python does not require classes to inherit from a shared base class to achieve polymorphism. As long as an object implements the required method (e.g. `.divide()`, `.get_reads()`, `.molecular_weight()`), Python calls it seamlessly.

#### Biological Example: Cellular Division
Different cells divide by fundamentally different mechanisms:
- A `ProkaryoticCell` divides by **Binary Fission**.
- A `EukaryoticCell` divides by **Mitosis**.
Yet a tissue or culture simulation can iterate over a list of cells and simply invoke `cell.divide()` on each!


In [ ]:
# Worked Example 4.5: Polymorphic cellular division
class ProkaryoticCell:
    def __init__(self, cell_id: str):
        self.cell_id = cell_id

    def divide(self) -> str:
        return f"Cell {self.cell_id} dividing via Binary Fission into two identical daughter cells."

class EukaryoticCell:
    def __init__(self, cell_id: str, ploidy: str = "diploid"):
        self.cell_id = cell_id
        self.ploidy = ploidy

    def divide(self) -> str:
        return f"Cell {self.cell_id} dividing via Mitosis (Prophase -> Metaphase -> Anaphase -> Telophase)."

# Polymorphic processing: a single loop processes different cell types uniformly
culture = [
    ProkaryoticCell("E_coli_01"),
    EukaryoticCell("HeLa_42", ploidy="aneuploid"),
    ProkaryoticCell("B_subtilis_09")
]

for cell in culture:
    print(cell.divide())


#### ✏️ Try it 4.5: Polymorphic Method Invocation
Add a new cell type `YeastCell` with a `.divide()` method returning `"Budding"` and verify that the simulation loop runs without modification.


In [ ]:
# Try it 4.5: Duck typing with Yeast
class YeastCell:
    def divide(self) -> str:
        return "Yeast cell dividing via Asymmetric Budding."

yeast = YeastCell()
assert "Budding" in yeast.divide()
print("Try it 4.5 passed ✔")


#### ⚠️ Common Mistakes with Polymorphism
- **Excessive `isinstance()` checks**: Writing `if isinstance(x, CellA): ... elif isinstance(x, CellB): ...` defeats the entire purpose of polymorphism! Just call `x.divide()`.
- **Inconsistent method signatures**: Giving one class `divide(self)` and another `divide(self, speed)` breaks uniform caller invocations.


### 4.6 Concept 6: Abstraction via Abstract Base Classes (`abc.ABC`)

**Abstraction** is the practice of exposing only the essential interface contract while hiding the implementation details.

In Python, formal abstraction is enforced using the `abc` (Abstract Base Classes) module:
```python
from abc import ABC, abstractmethod

class BiologicalSequence(ABC):
    @property
    @abstractmethod
    def alphabet(self) -> set[str]:
        """Must be defined by concrete subclasses."""
        pass

    @abstractmethod
    def molecular_weight(self) -> float:
        """Must be implemented by concrete subclasses."""
        pass
```

#### The Abstract Enforcement Contract
1. **Cannot be instantiated directly**: Attempting `s = BiologicalSequence()` raises:
   `TypeError: Can't instantiate abstract class BiologicalSequence with abstract methods`
2. **Subclasses MUST implement all abstract methods**: If a subclass fails to implement even one `@abstractmethod`, Python refuses to let that subclass be instantiated!
3. **Guarantees uniform API**: Ensures every sequence subclass (`DNA`, `RNA`, `Protein`) is guaranteed to implement `molecular_weight()`.


In [ ]:
# Worked Example 4.6: Abstract Base Class for biological sequences
class AbstractSequence(ABC):
    def __init__(self, seq_id: str, sequence: str):
        self.seq_id = seq_id
        self.sequence = sequence.strip().upper()
        # Enforce alphabet validation during initialization
        invalid = set(self.sequence) - self.alphabet
        if invalid:
            raise ValueError(f"Invalid base(s) {invalid} for {self.__class__.__name__}")

    @property
    @abstractmethod
    def alphabet(self) -> set[str]:
        """Allowed characters."""
        pass

    @abstractmethod
    def molecular_weight(self) -> float:
        """Compute molecular weight in Daltons."""
        pass

# Concrete DNA subclass
class DNA(AbstractSequence):
    @property
    def alphabet(self) -> set[str]:
        return {"A", "C", "G", "T", "N"}

    def molecular_weight(self) -> float:
        weights = {"A": 313.21, "C": 289.18, "G": 329.21, "T": 304.20, "N": 308.95}
        return round(sum(weights[b] for b in self.sequence), 2)

# 1. Attempting to instantiate the abstract class directly fails
try:
    AbstractSequence("test", "ATGC")
except TypeError as e:
    print(f"Direct ABC instantiation blocked: {e}")

# 2. Instantiating concrete subclass succeeds
dna_obj = DNA("d1", "ATGC")
print(f"DNA instance MW: {dna_obj.molecular_weight()} Da")


#### ✏️ Try it 4.6: Verify Incomplete Subclass Error
Create an incomplete subclass `IncompleteRNA(AbstractSequence)` that defines `alphabet` but forgets to implement `molecular_weight()`. Assert that instantiating it raises `TypeError`.


In [ ]:
# Try it 4.6: Incomplete subclass test
class IncompleteRNA(AbstractSequence):
    @property
    def alphabet(self):
        return {"A", "C", "G", "U"}

raised_type_error = False
try:
    IncompleteRNA("r1", "ACGU")
except TypeError:
    raised_type_error = True

assert raised_type_error, "Instantiating an incomplete ABC subclass must raise TypeError!"
print("Try it 4.6 passed ✔")


#### ⚠️ Common Mistakes with ABCs
- **Forgetting to inherit from `ABC`**: Using `@abstractmethod` on a class that does NOT inherit from `abc.ABC` does NOT enforce anything! Python will allow instantiation.
- **Abstract property syntax**: Remember to stack decorators:
  ```python
  @property
  @abstractmethod
  def prop(self): ...
  ```


### 4.7 Concept 7: Ten Core OOP Theory Questions (Self-Test & Interview Prep)

Test your conceptual mastery with these 10 core object-oriented software engineering questions:

#### Q1. Why does Python not truly enforce private access modifiers like C++ or Java?
> **Answer:** Python follows the core architectural philosophy: *"We are all consenting adults here."* Rigid compiler-enforced access levels add runtime complexity and hinder debugging, unit testing, and interactive experimentation. Python relies on naming conventions (`_protected`, `__private`) to signal intent rather than imposing rigid locks.

#### Q2. What is the fundamental difference between Abstraction and Encapsulation?
> **Answer:**
> - **Encapsulation** is about *information hiding and data integrity*: bundling data and methods together and restricting direct manipulation of internal state.
> - **Abstraction** is about *interface design and complexity reduction*: defining what an entity does (contract) without exposing how it does it (implementation).

#### Q3. How does name mangling work for double-underscore attributes, and why was it designed?
> **Answer:** When an identifier begins with two leading underscores (e.g. `self.__id`), Python transforms it into `_ClassName__id`. It was designed primarily to **prevent accidental attribute overriding** in subclasses during inheritance, not for security.

#### Q4. What is the purpose of `super()` compared to calling the parent class by name directly?
> **Answer:** `super()` dynamically determines the next class in the Method Resolution Order (MRO). Calling `Parent.__init__(self)` hardcodes the inheritance hierarchy and causes duplicate initializations or skipped classes in multiple inheritance ("diamond problem").

#### Q5. What is the Method Resolution Order (MRO), and how do you inspect it?
> **Answer:** MRO is the deterministic order in which Python searches for attributes and methods across a class hierarchy, computed via the **C3 Linearization Algorithm**. It can be inspected via `ClassName.mro()` or `ClassName.__mro__`.

#### Q6. What is "duck typing" and how does it relate to polymorphism in Python?
> **Answer:** Duck typing means an object's suitability is determined by the presence of specific methods and properties, rather than its inheritance hierarchy (*"If it walks like a duck and quacks like a duck, it's a duck"*). It enables dynamic polymorphism without formal interface declarations.

#### Q7. What happens if you try to instantiate a class that inherits from `abc.ABC` but fails to implement an `@abstractmethod`?
> **Answer:** Python raises a runtime `TypeError: Can't instantiate abstract class ... with abstract method ...`. The class remains abstract until all abstract methods are implemented.

#### Q8. Why is `@property` preferred in Python over explicit `get_x()` and `set_x()` methods?
> **Answer:** `@property` provides clean, intuitive attribute access syntax (`sample.concentration`) while seamlessly enabling validation, lazy evaluation, and read-only attributes without breaking existing public APIs.

#### Q9. What is the difference between composition ("has-a") and inheritance ("is-a") in biological modeling?
> **Answer:**
> - **Inheritance ("is-a")**: Subclassing when one entity is a specialized type of another (e.g., `Bacterium` *is an* `Organism`).
> - **Composition ("has-a")**: Embedding objects within another object when one entity contains or coordinates others (e.g., `Chromosome` *has many* `Gene` objects). Prefer composition over inheritance for flexible system architectures.

#### Q10. Can a class have multiple decorators on a method, and in what order are `@property` and custom decorators applied?
> **Answer:** Yes. Decorators evaluate from bottom to top (inside out). When combining `@property` with `@abstractmethod`, `@property` must be on the outside:
> ```python
> @property
> @abstractmethod
> def value(self): ...
> ```


---
## 5. Practice Exercises

Apply the four pillars of OOP and access modifiers to solve these problems:
- **Level 1 (★)**: Protected patient data and enzyme hierarchies.
- **Level 2 (★★)**: `@property` validation, polymorphic exporters, and MRO mixins.
- **Level 3 (★★★)**: Abstract base class variant caller architecture.

---


### Ex 3B.1 (★) — Secure Patient Clinical Sample Record

**Problem Statement:**  
Create a class `SecurePatientSample`:
1. `__init__(self, sample_id: str, patient_mrn: str, viral_load: float, collection_date: str)`:
   - `sample_id`: Public string.
   - `__patient_mrn`: **Private** Medical Record Number (name-mangled).
   - `_viral_load`: **Protected** float (copies/mL).
   - `collection_date`: Public string.
2. Property `viral_load`:
   - Getter returning `_viral_load`.
   - Setter that validates `val >= 0.0`. If negative, raises `ValueError`.
3. Method `get_masked_mrn(self) -> str`:
   - Returns MRN with only the last 3 digits visible (e.g. `MRN-987654` $\rightarrow$ `***-654`).

- **Input:** Sample and clinical parameters.
- **Output:** Encapsulated patient sample object.


In [ ]:
class SecurePatientSample:
    def __init__(self, sample_id: str, patient_mrn: str, viral_load: float, collection_date: str):
        self.sample_id = sample_id
        self.__patient_mrn = str(patient_mrn)
        self.collection_date = collection_date
        self.viral_load = viral_load  # triggers setter

    @property
    def viral_load(self) -> float:
        return self._viral_load

    @viral_load.setter
    def viral_load(self, value: float):
        val = float(value)
        if val < 0.0:
            raise ValueError(f"Viral load cannot be negative: {val}")
        self._viral_load = val

    def get_masked_mrn(self) -> str:
        last3 = self.__patient_mrn[-3:]
        return f"***-{last3}"


**Explanation:**
- `__patient_mrn` is private and mangled to `_SecurePatientSample__patient_mrn`.
- The setter for `viral_load` validates that biological concentration cannot be negative.
- `get_masked_mrn()` provides a controlled public view of private medical data.


In [ ]:
# ✅ Test Ex 3B.1
sample = SecurePatientSample("SMP_01", "MRN-554433", 12500.5, "2024-10-29")
assert sample.sample_id == "SMP_01"
assert sample.viral_load == 12500.5
assert sample.get_masked_mrn() == "***-433"

# Verify name mangling
assert "_SecurePatientSample__patient_mrn" in sample.__dict__
assert "__patient_mrn" not in sample.__dict__

# Test negative viral load validation
try:
    sample.viral_load = -10.0
    assert False, "Should raise ValueError on negative viral load"
except ValueError:
    pass

print("Ex 3B.1 passed ✔")


### Ex 3B.2 (★) — Enzyme Inheritance Hierarchy

**Problem Statement:**  
Create an inheritance hierarchy for biological enzymes:
1. Base class `Enzyme`:
   - `__init__(self, name: str, ec_number: str)`: stores `name` and `ec_number`.
   - Method `catalyze(self) -> str`: returns `"{name} (EC {ec_number}) is catalyzing a biochemical reaction."`
2. Subclass `RestrictionEndonuclease(Enzyme)`:
   - `__init__(self, name: str, ec_number: str, recognition_site: str)`: calls `super()`, stores uppercase `recognition_site`.
   - Overrides `catalyze(self) -> str`: returns `"{name} cleaves DNA at sequence {recognition_site}."`
3. Subclass `DNApolymerase(Enzyme)`:
   - `__init__(self, name: str, ec_number: str, error_rate: float)`: calls `super()`, stores `error_rate`.
   - Overrides `catalyze(self) -> str`: returns `"{name} synthesizing DNA with error rate {error_rate}."`

- **Input:** Enzyme parameters.
- **Output:** Hierarchical enzyme classes.


In [ ]:
class Enzyme:
    def __init__(self, name: str, ec_number: str):
        self.name = name
        self.ec_number = ec_number

    def catalyze(self) -> str:
        return f"{self.name} (EC {self.ec_number}) is catalyzing a biochemical reaction."

class RestrictionEndonuclease(Enzyme):
    def __init__(self, name: str, ec_number: str, recognition_site: str):
        super().__init__(name, ec_number)
        self.recognition_site = recognition_site.upper()

    def catalyze(self) -> str:
        return f"{self.name} cleaves DNA at sequence {self.recognition_site}."

class DNApolymerase(Enzyme):
    def __init__(self, name: str, ec_number: str, error_rate: float):
        super().__init__(name, ec_number)
        self.error_rate = error_rate

    def catalyze(self) -> str:
        return f"{self.name} synthesizing DNA with error rate {self.error_rate}."


**Explanation:**
- `super().__init__(name, ec_number)` delegates shared initialization to `Enzyme`.
- Method overriding in `catalyze()` models specialized catalytic behavior for distinct enzyme classes.


In [ ]:
# ✅ Test Ex 3B.2
bamhi = RestrictionEndonuclease("BamHI", "3.1.21.4", "GGATCC")
taq = DNApolymerase("Taq Polymerase", "2.7.7.7", 1e-4)

assert isinstance(bamhi, Enzyme)
assert isinstance(taq, Enzyme)
assert "GGATCC" in bamhi.catalyze()
assert "synthesizing DNA" in taq.catalyze()
print("Ex 3B.2 passed ✔")


### Ex 3B.3 (★★) — RNA Codon Property Validator

**Problem Statement:**  
Create a class `RNACodon`:
1. Class attribute `GENETIC_CODE`:
   Dictionary mapping 64 standard RNA codons to single-letter amino acid codes:
   `{"AUG": "M", "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L", "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S", "UAU": "Y", "UAC": "Y", "UAA": "*", "UAG": "*", "UGU": "C", "UGC": "C", "UGA": "*", "UGG": "W", "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L", "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P", "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q", "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R", "AUU": "I", "AUC": "I", "AUA": "I", "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T", "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K", "AGU": "S", "AGC": "S", "AGA": "R", "AGG": "R", "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V", "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A", "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E", "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G"}`
2. Property `codon`:
   - Setter validates:
     - Length must be exactly 3. If not, raises `ValueError`.
     - Characters must only be `A, C, G, U`. If not, raises `ValueError`.
     - Stores uppercase string in `_codon`.
   - Getter returns `_codon`.
3. Read-only property `amino_acid`:
   - Translates `_codon` using `GENETIC_CODE`.
4. Read-only property `is_stop`:
   - Returns `True` if `amino_acid == "*"`, else `False`.

- **Input:** Triplet RNA sequence.
- **Output:** Validated `RNACodon` object with computed translation properties.


In [ ]:
class RNACodon:
    GENETIC_CODE = {
        "AUG": "M", "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L",
        "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S", "UAU": "Y",
        "UAC": "Y", "UAA": "*", "UAG": "*", "UGU": "C", "UGC": "C",
        "UGA": "*", "UGG": "W", "CUU": "L", "CUC": "L", "CUA": "L",
        "CUG": "L", "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
        "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q", "CGU": "R",
        "CGC": "R", "CGA": "R", "CGG": "R", "AUU": "I", "AUC": "I",
        "AUA": "I", "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
        "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K", "AGU": "S",
        "AGC": "S", "AGA": "R", "AGG": "R", "GUU": "V", "GUC": "V",
        "GUA": "V", "GUG": "V", "GCU": "A", "GCC": "A", "GCA": "A",
        "GCG": "A", "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
        "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G"
    }

    def __init__(self, triplet: str):
        self.codon = triplet  # triggers setter

    @property
    def codon(self) -> str:
        return self._codon

    @codon.setter
    def codon(self, value: str):
        clean = value.strip().upper()
        if len(clean) != 3:
            raise ValueError(f"Codon must have length 3, got {len(clean)}: '{clean}'")
        if not set(clean).issubset({"A", "C", "G", "U"}):
            raise ValueError(f"Invalid RNA bases in codon: '{clean}'")
        self._codon = clean

    @property
    def amino_acid(self) -> str:
        return self.GENETIC_CODE[self._codon]

    @property
    def is_stop(self) -> bool:
        return self.amino_acid == "*"


**Explanation:**
- The setter guarantees that `_codon` is always a valid 3-character RNA triplet.
- `amino_acid` and `is_stop` are read-only properties derived dynamically from the genetic code dictionary.


In [ ]:
# ✅ Test Ex 3B.3
start_c = RNACodon("aug")
assert start_c.codon == "AUG"
assert start_c.amino_acid == "M"
assert not start_c.is_stop

stop_c = RNACodon("UAA")
assert stop_c.amino_acid == "*"
assert stop_c.is_stop

# Validation tests
try:
    RNACodon("AU")  # Too short
    assert False
except ValueError:
    pass

try:
    RNACodon("ATG")  # Contains T instead of U
    assert False
except ValueError:
    pass

print("Ex 3B.3 passed ✔")


### Ex 3B.4 (★★) — Polymorphic Sequence Exporters

**Problem Statement:**  
Create two exporter classes sharing a polymorphic interface:
1. `FastaExporter`:
   - Method `export(self, records: list[tuple[str, str]]) -> str`:
     - Takes list of `(seq_id, sequence)` tuples.
     - Returns formatted FASTA string with `>{seq_id}\n{sequence}\n`.
2. `TabDelimitedExporter`:
   - Method `export(self, records: list[tuple[str, str]]) -> str`:
     - Returns TSV string with header `"id\tsequence\n"` followed by rows `"{seq_id}\t{sequence}\n"`.
3. Standalone function `format_dataset(exporter, records: list[tuple[str, str]]) -> str`:
   - Demonstrates duck typing by calling `exporter.export(records)`.

- **Input:** Records list and exporter object.
- **Output:** Formatted exported string.


In [ ]:
class FastaExporter:
    def export(self, records: list[tuple[str, str]]) -> str:
        lines = []
        for seq_id, seq in records:
            lines.append(f">{seq_id}\n{seq}\n")
        return "".join(lines)

class TabDelimitedExporter:
    def export(self, records: list[tuple[str, str]]) -> str:
        lines = ["id\tsequence\n"]
        for seq_id, seq in records:
            lines.append(f"{seq_id}\t{seq}\n")
        return "".join(lines)

def format_dataset(exporter, records: list[tuple[str, str]]) -> str:
    """Polymorphic consumer relying on duck typing."""
    return exporter.export(records)


**Explanation:**
- `format_dataset` does not care about the concrete type of `exporter`; it relies entirely on duck typing to call `.export(records)`.
- This enables adding new formats (e.g. JSON, GenBank) without altering existing export pipelines.


In [ ]:
# ✅ Test Ex 3B.4
sample_recs = [("g1", "ATGC"), ("g2", "GGCC")]

fasta_out = format_dataset(FastaExporter(), sample_recs)
tsv_out = format_dataset(TabDelimitedExporter(), sample_recs)

assert ">g1\nATGC\n" in fasta_out
assert "id\tsequence\n" in tsv_out
assert "g1\tATGC\n" in tsv_out
print("Ex 3B.4 passed ✔")


### Ex 3B.5 (★★) — Multiple Inheritance & MRO with Mixins

**Problem Statement:**  
Create a mixin class to add sequencing quality analysis:
1. `QualityScoreMixin`:
   - `__init__(self, *args, quality_scores: list[int] = None, **kwargs)`:
     - Stores `self.quality_scores = quality_scores if quality_scores is not None else []`.
     - Forwards remaining `*args` and `**kwargs` via `super().__init__(*args, **kwargs)`.
   - Method `mean_quality(self) -> float`:
     - Returns arithmetic mean of `self.quality_scores`, rounded to 2 decimals (or `0.0` if empty).
2. `BaseSequence`:
   - `__init__(self, seq_id: str, seq: str)`: stores `seq_id` and `seq`.
3. `SequencingRead(QualityScoreMixin, BaseSequence)`:
   - Inherits from both `QualityScoreMixin` and `BaseSequence`.
   - `__init__(self, seq_id: str, seq: str, quality_scores: list[int] = None)`:
     - Uses `super()` to initialize cooperative multiple inheritance.

- **Input:** Sequence string, ID, and list of Phred quality scores.
- **Output:** Combined `SequencingRead` object with MRO resolution.


In [ ]:
class QualityScoreMixin:
    def __init__(self, *args, quality_scores: list[int] = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.quality_scores = quality_scores if quality_scores is not None else []

    def mean_quality(self) -> float:
        if not self.quality_scores:
            return 0.0
        return round(sum(self.quality_scores) / len(self.quality_scores), 2)

class BaseSequence:
    def __init__(self, seq_id: str, seq: str, **kwargs):
        super().__init__(**kwargs)
        self.seq_id = seq_id
        self.seq = seq.upper()

class SequencingRead(QualityScoreMixin, BaseSequence):
    def __init__(self, seq_id: str, seq: str, quality_scores: list[int] = None):
        super().__init__(seq_id=seq_id, seq=seq, quality_scores=quality_scores)


**Explanation:**
- Mixins provide reusable behavioral components without requiring rigid monolithic inheritance hierarchies.
- Using `*args`, `**kwargs`, and `super()` ensures that initialization chains cleanly across all parent classes in the MRO.


In [ ]:
# ✅ Test Ex 3B.5
read = SequencingRead("READ_001", "ATGCGT", quality_scores=[30, 32, 28, 40, 35, 33])
assert read.seq_id == "READ_001"
assert read.seq == "ATGCGT"
assert read.mean_quality() == 33.0

# Verify MRO
mro_names = [c.__name__ for c in SequencingRead.mro()]
assert mro_names == ["SequencingRead", "QualityScoreMixin", "BaseSequence", "object"]
print("Ex 3B.5 passed ✔")


### Ex 3B.6 (★★★) — Abstract Base Class Variant Caller Pipeline

**Problem Statement:**  
Create an extensible variant-calling architecture using Python's `abc` module:
1. Abstract base class `AbstractVariantCaller(ABC)`:
   - `__init__(self, caller_name: str)`: stores `caller_name`.
   - `@abstractmethod def call_variants(self, reference: str, sample: str) -> list[dict]`:
     - Abstract method contract that every concrete caller must implement.
2. Concrete class `SimpleSNPCaller(AbstractVariantCaller)`:
   - Implements `call_variants(self, reference: str, sample: str) -> list[dict]`:
     - Compares `reference` and `sample` position by position (assuming equal lengths).
     - If lengths differ, raises `ValueError`.
     - For each mismatch at index `i` (1-based coordinate):
       - Appends `{"position": i + 1, "ref": ref_base, "alt": alt_base}`.
     - Returns list of variant dictionaries.

- **Input:** Reference sequence and sample sequence strings.
- **Output:** Extracted single-nucleotide polymorphisms (SNPs).


In [ ]:
class AbstractVariantCaller(ABC):
    def __init__(self, caller_name: str):
        self.caller_name = caller_name

    @abstractmethod
    def call_variants(self, reference: str, sample: str) -> list[dict]:
        pass

class SimpleSNPCaller(AbstractVariantCaller):
    def call_variants(self, reference: str, sample: str) -> list[dict]:
        ref = reference.strip().upper()
        smp = sample.strip().upper()
        
        if len(ref) != len(smp):
            raise ValueError(f"Sequence lengths must match: ref={len(ref)}, sample={len(smp)}")
            
        variants = []
        for i, (r, a) in enumerate(zip(ref, smp)):
            if r != a:
                variants.append({
                    "position": i + 1,  # 1-based genomic coordinate
                    "ref": r,
                    "alt": a
                })
        return variants


**Explanation:**
- `AbstractVariantCaller` establishes a rigid contract enforcing the `call_variants()` interface across different variant caller algorithms (e.g. SNVs, Indels, Structural Variants).
- `SimpleSNPCaller` implements the contract, generating standardized 1-based genomic variant records.


In [ ]:
# ✅ Test Ex 3B.6
caller = SimpleSNPCaller("ToySNP")
assert caller.caller_name == "ToySNP"

# Test variant calling: Mismatch at pos 3 (G -> A) and pos 7 (T -> C)
ref_seq = "ATGCTTT"
smp_seq = "ATACTTC"
snps = caller.call_variants(ref_seq, smp_seq)

assert len(snps) == 2, f"Expected 2 SNPs, got {len(snps)}"
assert snps[0] == {"position": 3, "ref": "G", "alt": "A"}
assert snps[1] == {"position": 7, "ref": "T", "alt": "C"}

# Verify ABC blocks direct instantiation
try:
    AbstractVariantCaller("Abstract")
    assert False, "Should raise TypeError on instantiating ABC"
except TypeError:
    pass

print("Ex 3B.6 passed ✔")


---
## 6. Summary & Cheat Sheet

| OOP Pillar / Concept | Implementation Syntax | Biological Purpose |
|---|---|---|
| **Encapsulation** | `self._sequence` + `@property` | Protects biological invariants (e.g. valid nucleotide alphabet) |
| **Public Modifier** | `self.name` | Freely accessible gene names, coordinates |
| **Protected Modifier** | `self._name` | Internal convention: sub-methods, raw buffers |
| **Private Modifier** | `self.__name` (mangled to `_Class__name`) | Prevents subclass collision; protects patient IDs |
| **Property Getter** | `@property def gc(self):` | Clean attribute access (`seq.gc`) |
| **Property Setter** | `@seq.setter def seq(self, val):` | Validates new data before assignment |
| **Inheritance** | `class Sub(Parent): super().__init__()` | Code reuse for biological taxonomies |
| **Polymorphism** | `cell.divide()` | Uniform treatment of heterogeneous biological entities |
| **Duck Typing** | No base class required | If it implements `.export()`, it can be exported |
| **Abstraction** | `class ABCModel(ABC): @abstractmethod` | Defines interface contracts for pipelines |

---


## 7. Further Reading & Resources

- [Python Official Documentation: `abc` — Abstract Base Classes](https://docs.python.org/3/library/abc.html)
- [Real Python: Python's `@property` Decorator (A Guide to Getters & Setters)](https://realpython.com/python-property/)
- [Python Documentation: The C3 Method Resolution Order (MRO)](https://www.python.org/download/releases/2.3/mro/)
- [Martin Fowler: Composition Over Inheritance Architecture](https://martinfowler.com/)
